# Media Content Operations — Agent Instructions
### Industry-Specific Instructions & Sample Queries for Data Agents

This notebook defines **detailed agent instructions** for the Media (Content Operations) industry.
Run it **before** `06_Create_Data_Agent.ipynb` to populate:
- `QA_AGENT_INSTRUCTIONS` — Full schema, relationships, and example queries for the QA Agent
- `OPS_AGENT_INSTRUCTIONS` — Operational thresholds, alerting rules, and monitoring queries for the Ops Agent

Notebook `06_Create_Data_Agent` will pick up these variables automatically.
If this notebook is not run, `06` falls back to generic instructions.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT INSTRUCTIONS — Media Content Operations
# ════════════════════════════════════════════════════════════════════════

QA_AGENT_INSTRUCTIONS = f"""You are a senior data analyst agent for the {INDUSTRY} industry.
You answer ad-hoc questions about media content operations, production performance,
rights clearance, content quality, delivery workflows, crew wellness, and audience metrics
using the connected data sources.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables, full historical data
   Dimension tables:
   - dim_content_task_types: task_type_id, name, category, typical_hours, requires_approval, skill_level
   - dim_networks: network_id, name, type, region, audience_size, content_focus
   - dim_platforms: platform_id, name, type, region, subscriber_count, content_types_supported
   - dim_producers: producer_id, name, role, department, hire_date, specializations, credits_count, region
   - dim_rights_holders: holder_id, name, type, territory, contract_start, contract_end, content_categories
   - dim_shows: show_id, name, genre, network_id, season_count, episode_count, status, premiere_date, budget

   Batch fact tables:
   - fact_approval_workflows: workflow_id, show_id, producer_id, timestamp, stage, status, duration_hours, approver, revisions
   - fact_client_satisfaction: survey_id, network_id, producer_id, date, overall_score, timeliness_score, quality_score, communication_score, complaint_flag
   - fact_content_doc_events: event_id, producer_id, show_id, task_type_id, start_time, duration_minutes, status, error_count
   - fact_content_handoffs: handoff_id, show_id, from_producer_id, to_producer_id, timestamp, reason, doc_completeness_pct, pending_items
   - fact_content_quality: quality_id, show_id, producer_id, date, quality_score, technical_score, editorial_score, revision_count, defects_found
   - fact_crew_wellness: survey_id, producer_id, date, workload_score, overtime_hours, burnout_risk, admin_burden_perception, satisfaction_score

   Event fact tables:
   - fact_delivery_alerts: alert_id, show_id, platform_id, timestamp, alert_type, severity, deadline_date, days_remaining, status
   - fact_mam_interactions: interaction_id, producer_id, timestamp, asset_id, action, duration_seconds, application, search_count
   - fact_metadata_entries: entry_id, producer_id, show_id, timestamp, metadata_type, fields_completed, accuracy_score, duration_minutes
   - fact_production_performance: perf_id, show_id, date, budget_utilization_pct, schedule_adherence_pct, episodes_completed, episodes_planned
   - fact_regulatory_reviews: review_id, show_id, producer_id, timestamp, review_type, rating, compliance_flag, issues_found, region
   - fact_rights_clearance: clearance_id, show_id, holder_id, timestamp, content_type, status, turnaround_days, issues, territory

2. SEMANTIC MODEL ({SEMANTIC_MODEL_NAME}) — DirectQuery over the Warehouse.
3. LAKEHOUSE ({LAKEHOUSE_NAME}) — Delta tables for Spark-based analysis.
4. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time event and streaming data (KQL).
   Event tables: delivery_alerts, mam_interactions, metadata_entries, production_performance, regulatory_reviews, rights_clearance
   Streaming tables: audience_metrics, delivery_tracking, mam_activity, qc_results, rights_status

== KEY RELATIONSHIPS ==
- dim_producers.producer_id → fact tables on producer_id (also from_producer_id, to_producer_id)
- dim_shows.show_id → fact tables on show_id
- dim_networks.network_id → dim_shows, fact_client_satisfaction
- dim_platforms.platform_id → fact_delivery_alerts
- dim_rights_holders.holder_id → fact_rights_clearance
- dim_content_task_types.task_type_id → fact_content_doc_events

== EXAMPLE QUERIES ==

Q: Which shows have the best production performance?
→ Query fact_production_performance, GROUP BY show_id, AVG(schedule_adherence_pct), join dim_shows.

Q: Show content quality scores by producer.
→ Query fact_content_quality, GROUP BY producer_id, AVG(quality_score), join dim_producers.

Q: What is the rights clearance status?
→ Query fact_rights_clearance, GROUP BY status, COUNT(*), AVG(turnaround_days).

Q: Show client satisfaction by network.
→ Query fact_client_satisfaction, GROUP BY network_id, AVG(overall_score), join dim_networks.

Q: Which approval workflows are slowest?
→ Query fact_approval_workflows, GROUP BY stage, AVG(duration_hours), AVG(revisions).

Q: What are the content documentation times by task type?
→ Join fact_content_doc_events with dim_content_task_types, GROUP BY task type.

Q: Show content handoff patterns.
→ Query fact_content_handoffs, GROUP BY from_producer_id, COUNT(*), AVG(doc_completeness_pct).

Q: Which shows have the most regulatory issues?
→ Query fact_regulatory_reviews, GROUP BY show_id, SUM(issues_found), join dim_shows.

Q: Show delivery alerts by platform.
→ Query fact_delivery_alerts, GROUP BY platform_id, COUNT(*), join dim_platforms.

Q: What metadata accuracy scores do producers achieve?
→ Query fact_metadata_entries, GROUP BY producer_id, AVG(accuracy_score), join dim_producers.
"""

print(f"QA_AGENT_INSTRUCTIONS set ({len(QA_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT INSTRUCTIONS — Media Content Operations
# ════════════════════════════════════════════════════════════════════════

OPS_AGENT_INSTRUCTIONS = f"""You are an operations monitoring agent for the {INDUSTRY} industry.
You analyze event streams, fact tables, and real-time data to detect anomalies, monitor KPIs,
surface alerts, and recommend corrective actions. Focus on production schedules, delivery deadlines,
rights clearance, content quality, crew wellness, and audience metrics.

== DATA SOURCES ==

1. WAREHOUSE ({WAREHOUSE_NAME}) — SQL tables for historical analysis.
   Key operational tables:
   - fact_crew_wellness: CRITICAL if burnout_risk = true, workload_score > 8
   - fact_delivery_alerts: CRITICAL if days_remaining < 3, severity = 'Critical'
   - fact_production_performance: budget_utilization_pct > 100 (over budget), schedule_adherence_pct < 80
   - fact_content_quality: quality_score < 70, defects_found > 5
   - fact_rights_clearance: status = 'Blocked', turnaround_days > 14
   - fact_approval_workflows: status = 'Blocked', duration_hours > 48
   - fact_client_satisfaction: overall_score < 5, complaint_flag = true
   - fact_regulatory_reviews: compliance_flag = false
   - fact_content_handoffs: doc_completeness_pct < 80%, pending_items > 5
   - fact_content_doc_events: error_count > 3, duration > 45 min
   - fact_metadata_entries: accuracy_score < 80
   - fact_mam_interactions: search_count > 10 (frustration indicator)

   Dimension tables: dim_producers, dim_shows, dim_networks, dim_platforms, dim_rights_holders, dim_content_task_types

2. KQL DATABASE ({EVENTHOUSE_DATABASE}) — Real-time and streaming data.
   Streaming tables:
   - audience_metrics: metric_id, show_id, timestamp, platform_id, viewers, engagement_rate, sentiment_score
     → Alert on sentiment_score < 3, engagement_rate drop > 20%
   - delivery_tracking: tracking_id, show_id, timestamp, platform_id, milestone, status, pct_complete
     → Alert on status = 'Delayed'
   - mam_activity: activity_id, producer_id, timestamp, asset_id, action, application
   - qc_results: result_id, show_id, timestamp, check_type, pass_flag, defects, severity
     → Alert on pass_flag = false, severity = 'Critical'
   - rights_status: status_id, show_id, timestamp, holder_id, content_type, status, territory
     → Alert on status = 'Blocked' or 'Expired'

== ALERTING THRESHOLDS ==
- Burnout: burnout_risk = true, workload_score > 8, overtime_hours > 20
- Delivery: days_remaining < 3 (critical), < 7 (warning)
- Budget: budget_utilization_pct > 100%
- Quality: quality_score < 70, defects_found > 5
- Rights: status = 'Blocked', turnaround > 14 days
- Satisfaction: overall_score < 5, complaint_flag = true
- QC: pass_flag = false, severity = 'Critical'
- Audience: sentiment < 3, engagement drop > 20%

== EXAMPLE QUERIES ==

Q: Which crew members are at burnout risk?
→ Query fact_crew_wellness WHERE burnout_risk = true, join dim_producers.

Q: What delivery deadlines are approaching?
→ Query fact_delivery_alerts WHERE days_remaining < 7, join dim_shows, dim_platforms.

Q: Which shows are over budget?
→ Query fact_production_performance WHERE budget_utilization_pct > 100, join dim_shows.

Q: Show real-time QC failures.
→ KQL: qc_results | where pass_flag == false | project show_id, check_type, defects, severity

Q: Which rights are blocked?
→ KQL: rights_status | where status == 'Blocked' | project show_id, holder_id, content_type, territory

Q: Show audience sentiment trends.
→ KQL: audience_metrics | where timestamp > ago(24h) | summarize avg(sentiment_score) by show_id

Q: Show content quality issues.
→ Query fact_content_quality WHERE quality_score < 70, join dim_shows.

Q: What delivery tracking delays exist?
→ KQL: delivery_tracking | where status == 'Delayed' | project show_id, platform_id, milestone, pct_complete

Q: Show client complaints.
→ Query fact_client_satisfaction WHERE complaint_flag = true, join dim_networks.

Q: Which approval workflows are blocked?
→ Query fact_approval_workflows WHERE status = 'Blocked', join dim_shows.
"""

print(f"OPS_AGENT_INSTRUCTIONS set ({len(OPS_AGENT_INSTRUCTIONS)} chars).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# PER-DATASOURCE INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════

QA_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse contains all media content operations data as SQL tables.

DIMENSION TABLES:
- dim_content_task_types, dim_networks, dim_platforms, dim_producers, dim_rights_holders, dim_shows

FACT TABLES:
- fact_approval_workflows, fact_client_satisfaction, fact_content_doc_events, fact_content_handoffs,
  fact_content_quality, fact_crew_wellness

EVENT FACT TABLES:
- fact_delivery_alerts, fact_mam_interactions, fact_metadata_entries, fact_production_performance,
  fact_regulatory_reviews, fact_rights_clearance

KEY JOINS:
- dim_producers.producer_id → fact tables on producer_id
- dim_shows.show_id → fact tables on show_id
- dim_networks.network_id → dim_shows, fact_client_satisfaction
- dim_platforms.platform_id → fact_delivery_alerts
- dim_rights_holders.holder_id → fact_rights_clearance""",

    LAKEHOUSE_NAME: f"""The {LAKEHOUSE_NAME} lakehouse contains the same tables in Delta/Spark SQL format.
Same schema and joins as the Warehouse. Use LIMIT instead of TOP for row limits.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database contains real-time event and streaming tables.

EVENT TABLES (KQL):
- delivery_alerts, mam_interactions, metadata_entries, production_performance, regulatory_reviews, rights_clearance

STREAMING TABLES:
- audience_metrics, delivery_tracking, mam_activity, qc_results, rights_status

Use KQL syntax (not SQL).""",
}

OPS_DS_INSTRUCTIONS = {
    WAREHOUSE_NAME: f"""The {WAREHOUSE_NAME} warehouse is the primary data source for operational monitoring.

KEY OPERATIONAL TABLES AND THRESHOLDS:
- fact_crew_wellness: CRITICAL if burnout_risk = true, workload_score > 8
- fact_delivery_alerts: days_remaining < 3 (critical), < 7 (warning)
- fact_production_performance: budget_utilization_pct > 100, schedule_adherence_pct < 80
- fact_content_quality: quality_score < 70, defects_found > 5
- fact_rights_clearance: status = 'Blocked', turnaround_days > 14
- fact_client_satisfaction: overall_score < 5, complaint_flag = true

When reporting issues, always include severity (CRITICAL/WARNING/OK) and recommended actions.""",

    EVENTHOUSE_DATABASE: f"""The {EVENTHOUSE_DATABASE} KQL database provides real-time telemetry.

STREAMING ALERTS:
- qc_results: pass_flag = false, severity = 'Critical'
- rights_status: status = 'Blocked' or 'Expired'
- audience_metrics: sentiment_score < 3
- delivery_tracking: status = 'Delayed'

Use KQL syntax. Prefer summarize/count/avg for aggregations. Use ago(24h) for recent activity.""",
}

print(f"QA_DS_INSTRUCTIONS set: {', '.join(QA_DS_INSTRUCTIONS.keys())}")
print(f"OPS_DS_INSTRUCTIONS set: {', '.join(OPS_DS_INSTRUCTIONS.keys())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# QA AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

QA_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which shows have the best production performance?",
            "query": """SELECT s.name AS show, s.genre,\n       AVG(pp.schedule_adherence_pct) AS avg_adherence,\n       AVG(pp.budget_utilization_pct) AS avg_budget,\n       SUM(pp.episodes_completed) AS completed\nFROM fact_production_performance pp\nJOIN dim_shows s ON pp.show_id = s.show_id\nGROUP BY s.name, s.genre\nORDER BY avg_adherence DESC"""
        },
        {
            "question": "Show content quality scores by producer.",
            "query": """SELECT p.name, p.role,\n       AVG(cq.quality_score) AS avg_quality,\n       AVG(cq.technical_score) AS avg_technical,\n       AVG(cq.editorial_score) AS avg_editorial,\n       SUM(cq.defects_found) AS total_defects\nFROM fact_content_quality cq\nJOIN dim_producers p ON cq.producer_id = p.producer_id\nGROUP BY p.name, p.role\nORDER BY avg_quality DESC"""
        },
        {
            "question": "What is the rights clearance status?",
            "query": """SELECT rc.status,\n       COUNT(*) AS clearances,\n       AVG(rc.turnaround_days) AS avg_turnaround,\n       SUM(CASE WHEN rc.issues IS NOT NULL THEN 1 ELSE 0 END) AS with_issues\nFROM fact_rights_clearance rc\nGROUP BY rc.status\nORDER BY clearances DESC"""
        },
        {
            "question": "Show client satisfaction by network.",
            "query": """SELECT n.name AS network, n.type,\n       AVG(cs.overall_score) AS avg_overall,\n       AVG(cs.timeliness_score) AS avg_timeliness,\n       AVG(cs.quality_score) AS avg_quality,\n       SUM(CASE WHEN cs.complaint_flag = true THEN 1 ELSE 0 END) AS complaints\nFROM fact_client_satisfaction cs\nJOIN dim_networks n ON cs.network_id = n.network_id\nGROUP BY n.name, n.type\nORDER BY avg_overall DESC"""
        },
        {
            "question": "Which approval workflows are slowest?",
            "query": """SELECT aw.stage,\n       COUNT(*) AS workflows,\n       AVG(aw.duration_hours) AS avg_duration,\n       AVG(aw.revisions) AS avg_revisions\nFROM fact_approval_workflows aw\nGROUP BY aw.stage\nORDER BY avg_duration DESC"""
        },
        {
            "question": "Show content documentation by task type.",
            "query": """SELECT ct.name AS task_type, ct.category,\n       COUNT(*) AS events,\n       AVG(cde.duration_minutes) AS avg_duration,\n       AVG(cde.error_count) AS avg_errors\nFROM fact_content_doc_events cde\nJOIN dim_content_task_types ct ON cde.task_type_id = ct.task_type_id\nGROUP BY ct.name, ct.category\nORDER BY avg_duration DESC"""
        },
        {
            "question": "Show content handoff patterns.",
            "query": """SELECT p.name AS from_producer,\n       COUNT(*) AS handoffs,\n       AVG(ch.doc_completeness_pct) AS avg_completeness,\n       AVG(ch.pending_items) AS avg_pending\nFROM fact_content_handoffs ch\nJOIN dim_producers p ON ch.from_producer_id = p.producer_id\nGROUP BY p.name\nORDER BY avg_completeness ASC"""
        },
        {
            "question": "Which shows have regulatory issues?",
            "query": """SELECT s.name AS show, rr.review_type, rr.region,\n       SUM(rr.issues_found) AS total_issues,\n       SUM(CASE WHEN rr.compliance_flag = false THEN 1 ELSE 0 END) AS non_compliant\nFROM fact_regulatory_reviews rr\nJOIN dim_shows s ON rr.show_id = s.show_id\nGROUP BY s.name, rr.review_type, rr.region\nORDER BY total_issues DESC"""
        },
        {
            "question": "Show delivery alerts by platform.",
            "query": """SELECT pl.name AS platform, da.alert_type, da.severity,\n       COUNT(*) AS alerts,\n       AVG(da.days_remaining) AS avg_days_remaining\nFROM fact_delivery_alerts da\nJOIN dim_platforms pl ON da.platform_id = pl.platform_id\nGROUP BY pl.name, da.alert_type, da.severity\nORDER BY avg_days_remaining ASC"""
        },
        {
            "question": "Show metadata accuracy by producer.",
            "query": """SELECT p.name, p.role,\n       AVG(me.accuracy_score) AS avg_accuracy,\n       AVG(me.fields_completed) AS avg_fields,\n       AVG(me.duration_minutes) AS avg_duration\nFROM fact_metadata_entries me\nJOIN dim_producers p ON me.producer_id = p.producer_id\nGROUP BY p.name, p.role\nORDER BY avg_accuracy DESC"""
        },
    ],

    LAKEHOUSE_NAME: [
        {
            "question": "Which producers are at burnout risk?",
            "query": """SELECT p.name, p.role,\n       cw.workload_score, cw.overtime_hours, cw.burnout_risk\nFROM fact_crew_wellness cw\nJOIN dim_producers p ON cw.producer_id = p.producer_id\nWHERE cw.burnout_risk = true\nORDER BY cw.workload_score DESC"""
        },
        {
            "question": "Show rights clearance by territory.",
            "query": """SELECT territory, status,\n       COUNT(*) AS clearances,\n       AVG(turnaround_days) AS avg_days\nFROM fact_rights_clearance\nGROUP BY territory, status\nORDER BY avg_days DESC"""
        },
        {
            "question": "Show shows by genre and budget.",
            "query": """SELECT genre, COUNT(*) AS shows,\n       AVG(budget) AS avg_budget,\n       SUM(episode_count) AS total_episodes\nFROM dim_shows\nGROUP BY genre\nORDER BY avg_budget DESC"""
        },
        {
            "question": "Show MAM interaction patterns.",
            "query": """SELECT action, application,\n       COUNT(*) AS interactions,\n       AVG(duration_seconds) AS avg_duration\nFROM fact_mam_interactions\nGROUP BY action, application\nORDER BY interactions DESC\nLIMIT 20"""
        },
        {
            "question": "Show production schedule adherence.",
            "query": """SELECT s.name AS show,\n       AVG(pp.schedule_adherence_pct) AS adherence\nFROM fact_production_performance pp\nJOIN dim_shows s ON pp.show_id = s.show_id\nGROUP BY s.name\nORDER BY adherence ASC\nLIMIT 20"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show audience sentiment by show.",
            "query": """audience_metrics\n| where timestamp > ago(24h)\n| summarize avg_sentiment = avg(sentiment_score), avg_engagement = avg(engagement_rate) by show_id\n| order by avg_sentiment asc"""
        },
        {
            "question": "Show QC results.",
            "query": """qc_results\n| where timestamp > ago(24h)\n| summarize passed = countif(pass_flag == true), failed = countif(pass_flag == false) by check_type\n| order by failed desc"""
        },
        {
            "question": "What rights status changes occurred?",
            "query": """rights_status\n| where timestamp > ago(24h)\n| summarize count() by status, content_type\n| order by count_ desc"""
        },
        {
            "question": "Show delivery tracking progress.",
            "query": """delivery_tracking\n| where timestamp > ago(24h)\n| project show_id, platform_id, milestone, status, pct_complete\n| order by pct_complete asc"""
        },
        {
            "question": "Show MAM activity patterns.",
            "query": """mam_activity\n| where timestamp > ago(24h)\n| summarize actions = count() by producer_id, action\n| order by actions desc"""
        },
    ],
}

print(f"QA_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in QA_FEWSHOTS.items())}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# OPS AGENT — PER-DATASOURCE EXAMPLE QUERIES (fewshots)
# ════════════════════════════════════════════════════════════════════════

OPS_FEWSHOTS = {
    WAREHOUSE_NAME: [
        {
            "question": "Which crew members are at burnout risk?",
            "query": """SELECT p.name, p.role, p.department,\n       cw.workload_score, cw.overtime_hours,\n       cw.burnout_risk, cw.satisfaction_score\nFROM fact_crew_wellness cw\nJOIN dim_producers p ON cw.producer_id = p.producer_id\nWHERE cw.burnout_risk = true\nORDER BY cw.workload_score DESC"""
        },
        {
            "question": "What delivery deadlines are approaching?",
            "query": """SELECT s.name AS show, pl.name AS platform,\n       da.deadline_date, da.days_remaining, da.severity,\n       CASE WHEN da.days_remaining < 3 THEN 'CRITICAL'\n            WHEN da.days_remaining < 7 THEN 'WARNING'\n            ELSE 'OK' END AS alert_level\nFROM fact_delivery_alerts da\nJOIN dim_shows s ON da.show_id = s.show_id\nJOIN dim_platforms pl ON da.platform_id = pl.platform_id\nWHERE da.days_remaining < 7\nORDER BY da.days_remaining ASC"""
        },
        {
            "question": "Which shows are over budget?",
            "query": """SELECT s.name AS show, s.genre,\n       pp.budget_utilization_pct, pp.schedule_adherence_pct,\n       pp.episodes_completed, pp.episodes_planned\nFROM fact_production_performance pp\nJOIN dim_shows s ON pp.show_id = s.show_id\nWHERE pp.budget_utilization_pct > 100\nORDER BY pp.budget_utilization_pct DESC"""
        },
        {
            "question": "Show content quality issues.",
            "query": """SELECT s.name AS show, p.name AS producer,\n       cq.quality_score, cq.defects_found, cq.revision_count\nFROM fact_content_quality cq\nJOIN dim_shows s ON cq.show_id = s.show_id\nJOIN dim_producers p ON cq.producer_id = p.producer_id\nWHERE cq.quality_score < 70\nORDER BY cq.quality_score ASC"""
        },
        {
            "question": "Which rights clearances are blocked?",
            "query": """SELECT s.name AS show, rh.name AS rights_holder,\n       rc.content_type, rc.territory, rc.turnaround_days, rc.issues\nFROM fact_rights_clearance rc\nJOIN dim_shows s ON rc.show_id = s.show_id\nJOIN dim_rights_holders rh ON rc.holder_id = rh.holder_id\nWHERE rc.status = 'Blocked'\nORDER BY rc.turnaround_days DESC"""
        },
        {
            "question": "Show blocked approval workflows.",
            "query": """SELECT s.name AS show, aw.stage, aw.approver,\n       aw.duration_hours, aw.revisions\nFROM fact_approval_workflows aw\nJOIN dim_shows s ON aw.show_id = s.show_id\nWHERE aw.status = 'Blocked'\nORDER BY aw.duration_hours DESC"""
        },
        {
            "question": "Show client complaints.",
            "query": """SELECT n.name AS network,\n       cs.overall_score, cs.timeliness_score, cs.quality_score\nFROM fact_client_satisfaction cs\nJOIN dim_networks n ON cs.network_id = n.network_id\nWHERE cs.complaint_flag = true\nORDER BY cs.overall_score ASC"""
        },
        {
            "question": "Show non-compliant regulatory reviews.",
            "query": """SELECT s.name AS show, rr.review_type, rr.region,\n       rr.issues_found, rr.rating\nFROM fact_regulatory_reviews rr\nJOIN dim_shows s ON rr.show_id = s.show_id\nWHERE rr.compliance_flag = false\nORDER BY rr.issues_found DESC"""
        },
        {
            "question": "Show content handoffs with incomplete docs.",
            "query": """SELECT p.name AS from_producer,\n       ch.doc_completeness_pct, ch.pending_items, ch.reason\nFROM fact_content_handoffs ch\nJOIN dim_producers p ON ch.from_producer_id = p.producer_id\nWHERE ch.doc_completeness_pct < 80\nORDER BY ch.doc_completeness_pct ASC"""
        },
        {
            "question": "Show schedule behind shows.",
            "query": """SELECT s.name AS show,\n       pp.schedule_adherence_pct, pp.episodes_completed, pp.episodes_planned\nFROM fact_production_performance pp\nJOIN dim_shows s ON pp.show_id = s.show_id\nWHERE pp.schedule_adherence_pct < 80\nORDER BY pp.schedule_adherence_pct ASC"""
        },
    ],

    EVENTHOUSE_DATABASE: [
        {
            "question": "Show real-time QC failures.",
            "query": """qc_results\n| where pass_flag == false\n| project show_id, check_type, defects, severity\n| order by timestamp desc"""
        },
        {
            "question": "Which rights are blocked or expired?",
            "query": """rights_status\n| where status in ('Blocked', 'Expired')\n| project show_id, holder_id, content_type, status, territory\n| order by timestamp desc"""
        },
        {
            "question": "Show audience sentiment drops.",
            "query": """audience_metrics\n| where sentiment_score < 3\n| project show_id, platform_id, viewers, sentiment_score, engagement_rate\n| order by sentiment_score asc"""
        },
        {
            "question": "Show delivery tracking delays.",
            "query": """delivery_tracking\n| where status == 'Delayed'\n| project show_id, platform_id, milestone, pct_complete\n| order by pct_complete asc"""
        },
        {
            "question": "Show MAM activity volume.",
            "query": """mam_activity\n| where timestamp > ago(24h)\n| summarize actions = count() by producer_id, action\n| order by actions desc"""
        },
        {
            "question": "Show audience engagement trend.",
            "query": """audience_metrics\n| where timestamp > ago(7d)\n| summarize avg_engagement = avg(engagement_rate), avg_viewers = avg(viewers) by bin(timestamp, 1d), show_id\n| order by timestamp asc"""
        },
        {
            "question": "Show QC failure trend.",
            "query": """qc_results\n| where timestamp > ago(7d)\n| summarize passed = countif(pass_flag == true), failed = countif(pass_flag == false) by bin(timestamp, 1d)\n| order by timestamp asc"""
        },
    ],
}

print(f"OPS_FEWSHOTS set: {', '.join(f'{k}: {len(v)} queries' for k, v in OPS_FEWSHOTS.items())}")

## Sample Questions for the Media Data Agents

### QA Agent — `Media_QA_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Performance | Which shows have the best production performance? |
| 2 | Quality | Show content quality scores by producer. |
| 3 | Rights | What is the rights clearance status? |
| 4 | Satisfaction | Show client satisfaction by network. |
| 5 | Approvals | Which approval workflows are slowest? |
| 6 | Documentation | Show content documentation by task type. |
| 7 | Handoffs | Show content handoff patterns. |
| 8 | Regulatory | Which shows have regulatory issues? |
| 9 | Delivery | Show delivery alerts by platform. |
| 10 | Metadata | Show metadata accuracy by producer. |
| 11 | Wellness | Which producers are at burnout risk? |
| 12 | Audience | Show audience sentiment by show. |
| 13 | QC | Show QC results. |
| 14 | Genre | Show shows by genre and budget. |
| 15 | MAM | Show MAM interaction patterns. |

### Ops Agent — `Media_Ops_Agent`

| # | Category | Sample Question |
|---|----------|------------------|
| 1 | Burnout | Which crew members are at burnout risk? |
| 2 | Delivery | What delivery deadlines are approaching? |
| 3 | Budget | Which shows are over budget? |
| 4 | Quality | Show content quality issues. |
| 5 | Rights | Which rights clearances are blocked? |
| 6 | Approvals | Show blocked approval workflows. |
| 7 | Satisfaction | Show client complaints. |
| 8 | Regulatory | Show non-compliant regulatory reviews. |
| 9 | Handoffs | Show content handoffs with incomplete docs. |
| 10 | Schedule | Show schedule behind shows. |
| 11 | QC | Show real-time QC failures. |
| 12 | Audience | Show audience sentiment drops. |
| 13 | Tracking | Show delivery tracking delays. |
| 14 | Trends | Show audience engagement trend. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# SAVE INSTRUCTIONS FOR PARENT NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

import json

_result = json.dumps({
    "qa": QA_AGENT_INSTRUCTIONS,
    "ops": OPS_AGENT_INSTRUCTIONS,
    "qa_fewshots": QA_FEWSHOTS,
    "ops_fewshots": OPS_FEWSHOTS,
    "qa_ds_instructions": QA_DS_INSTRUCTIONS,
    "ops_ds_instructions": OPS_DS_INSTRUCTIONS
})

_tmp_path = "file:/lakehouse/default/Files/_temp/agent_instructions.json"
notebookutils.fs.put(_tmp_path, _result, True)
print(f"  Written {len(_result):,} bytes to {_tmp_path}")

print(f"{'═'*60}")
print(f"AGENT INSTRUCTIONS LOADED — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  QA_AGENT_INSTRUCTIONS:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
print(f"  OPS_AGENT_INSTRUCTIONS: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
print(f"  QA_FEWSHOTS:  {sum(len(v) for v in QA_FEWSHOTS.values())} total queries across {len(QA_FEWSHOTS)} datasources")
print(f"  OPS_FEWSHOTS: {sum(len(v) for v in OPS_FEWSHOTS.values())} total queries across {len(OPS_FEWSHOTS)} datasources")

notebookutils.notebook.exit("ok")